<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>10. Agrupaciones</font>  </h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>

---

En esta libreta se trabajará con una versión del conjunto de datos `titanic.csv` en la que se han cambiado los nombres de las columnas y se han recodificado algunos valores.  

In [1]:
import pandas as pd

df_titanic = pd.read_csv('data/titanic_prep.csv', index_col=0)
df_titanic.head()

,Sobrevive,Clase,Nombre,Sexo,Edad,Esposa/Hermanos,Padres/Hijos,Ticket,Tarifa,Número de Cabina,Puerto
Pasajero,,,,,,,,,,,
1,No,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,Yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,Yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,Yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,No,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



---
## <font color="#5E5E5E">1. Agrupación por categorías mediante `DataFrame.groupby()` </font>

Una de las tareas que más frecuentes cuando los `DataFrame` contienen columnas categóricas es el análisis por grupos. Básicamente consiste en agrupar las filas en función de una categoría (valor en una o varias columnas, índice, u obtenido a partir de éstos), y obtener y comparar los datos de cada grupo. 

La gestión de los grupos por categorías puede hacerse mediante selección condicional. La siguiente celda muestra el precio medio de la tarifa para cada una de las clases.

In [2]:
for clase in df_titanic['Clase'].unique():
    print(f"{clase}: {df_titanic[df_titanic['Clase']==clase]['Tarifa'].mean():.2f}")

third: 13.68
first: 84.15
second: 20.66


Sin embargo, *pandas* implementa funciones muy potentes y versátiles en relación a este tipo de análisis. 

El método `DataFrame.groupby(col)` agrupa las filas del `DataFrame` en función de los valores que contiene la columna o columnas `col`. Como resultado, produce una estructura en la que **las operaciones, se aplican a nivel de grupo**. Por ejemplo, la siguiente celda reproduce el resultado del ejemplo anterior, proporcinando la media por columna para cada categoría. El resultado final es un `DataFrame`. Además, mediante el parámetro `sort=True` (valor por defecto) puede indicarse que se ordenen las etiquetas del índice de la estructura resultante.

In [4]:
#df_titanic.mean()
df_titanic.groupby('Clase', sort=True).mean(numeric_only=True)

,Edad,Esposa/Hermanos,Padres/Hijos,Tarifa
Clase,,,,
first,38.233441,0.416667,0.356481,84.154687
second,29.877630,0.402174,0.380435,20.662183
third,25.140620,0.615071,0.393075,13.675550


En el ejemplo, se ha pasado el nombre de una columna para indicar al método **la** `Series` **de datos utilizada para hacer la agrupación**. A este respecto, el método `DataFrame.groupby()`contempla cuatro posibilidades que se describen a continuación.

#### <font color="#5E5E5E"> 1) Agrupación por nombre de columna o nombres de columnas.</font>

`DataFrame.groupby(col)` puede recibir como argumento el nombre de una columna (ejemplo anterior) o varios nombres de columnas. En este último caso, el resultado  de la agregación posterior es una `Series` o `DataFrame` con un índice jerárquico de varios niveles (se verán después).

In [5]:
df_titanic.groupby(['Puerto','Clase']).mean(numeric_only=True)

Edad  Esposa/Hermanos  Padres/Hijos      Tarifa
Puerto Clase                                                       
C      first   38.027027         0.376471      0.317647  104.718529
       second  22.766667         0.588235      0.529412   25.358335
       third   20.741951         0.348485      0.378788   11.214083
Q      first   38.500000         1.500000      0.000000   90.000000
       second  43.500000         0.000000      0.000000   12.350000
       third   25.937500         0.416667      0.180556   11.183393
S      first   38.152037         0.433071      0.393701   70.364862
       second  30.386731         0.390244      0.371951   20.327439
       third   25.696552         0.705382      0.439093   14.644083

El parámetro `as_index=False` permite indicar que las claves aparezcan como columnas de la estructura agregada, en lugar de en el índice.

In [6]:
df_titanic.groupby(['Puerto','Clase'], as_index=False).mean(numeric_only=True)
# df_titanic.groupby(['Puerto','Clase']).mean().reset_index() # Sería equivalente.

,Puerto,Clase,Edad,Esposa/Hermanos,Padres/Hijos,Tarifa
0,C,first,38.027027,0.376471,0.317647,104.718529
1,C,second,22.766667,0.588235,0.529412,25.358335
2,C,third,20.741951,0.348485,0.378788,11.214083
3,Q,first,38.500000,1.500000,0.000000,90.000000
4,Q,second,43.500000,0.000000,0.000000,12.350000
5,Q,third,25.937500,0.416667,0.180556,11.183393
6,S,first,38.152037,0.433071,0.393701,70.364862
7,S,second,30.386731,0.390244,0.371951,20.327439
8,S,third,25.696552,0.705382,0.439093,14.644083


#### <font color="#5E5E5E"> 2) Agrupación por *Serie*.</font>

`DataFrame.groupby()` permite agrupar los valores en función de una `Series`.  En realidad, **es equivalente al uso anterior** cuando se proporciona **un solo nombre** de columna.

In [7]:
df_titanic.groupby(df_titanic['Clase']).mean(numeric_only=True)

,Edad,Esposa/Hermanos,Padres/Hijos,Tarifa
Clase,,,,
first,38.233441,0.416667,0.356481,84.154687
second,29.877630,0.402174,0.380435,20.662183
third,25.140620,0.615071,0.393075,13.675550


<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Utilizar la columna de manera explícita como argumento no aporta nada con respecto a hacerlo con el nombre de la columna. Sin embargo, ayuda a entender funcionalidades más complejas que se verán después 
</div>

#### <font color="#5E5E5E"> 3) Agrupación por valor del índice.</font>

También es posible agrupar las filas según el valor del índice. Mediante el argumento `level` se puede proporcionar el nombre o nivel del índice (que en el caso de los índices de un solo nivel es 0). 


Para ilustrar este ejemplo, se utilizará un `DataFrame` con los datos obtenidos en una agrupación anterior.

In [8]:
df_grouped = df_titanic.groupby(['Puerto','Clase']).mean(numeric_only=True) # Sea crea el DataFrame
df_grouped

Edad  Esposa/Hermanos  Padres/Hijos      Tarifa
Puerto Clase                                                       
C      first   38.027027         0.376471      0.317647  104.718529
       second  22.766667         0.588235      0.529412   25.358335
       third   20.741951         0.348485      0.378788   11.214083
Q      first   38.500000         1.500000      0.000000   90.000000
       second  43.500000         0.000000      0.000000   12.350000
       third   25.937500         0.416667      0.180556   11.183393
S      first   38.152037         0.433071      0.393701   70.364862
       second  30.386731         0.390244      0.371951   20.327439
       third   25.696552         0.705382      0.439093   14.644083

Este `DataFrame`, que está indexado con un índice de dos niveles (se verá después), puede indexarse por cualquiera de esos dos niveles. 

In [9]:
df_grouped.groupby(level='Puerto').mean(numeric_only=True)  # Muestra la medias medias para cada puerto. Usa el nombre del índice.
#df_grouped.groupby(level=1).mean(numeric_only=True)        # Muestra las medias para cada clase. Usa el nivel del índice.

,Edad,Esposa/Hermanos,Padres/Hijos,Tarifa
Puerto,,,,
C,27.178548,0.437730,0.408616,47.096983
Q,35.979167,0.638889,0.060185,37.844464
S,31.411773,0.509566,0.401582,35.112128


#### <font color="#5E5E5E"> 4) Función sobre el índice.</font>

Por último, `DataFrame.groupby()` puede recibir una función como parámetro, y la aplica sobre cada etiqueta del índice. La agrupación se hace sobre el resultado de esta transformación. 

In [10]:
df_titanic.groupby(lambda p:p%2).mean(numeric_only=True) # Como el índice es un entero identificador, agrupa entre pares e impares.

,Edad,Esposa/Hermanos,Padres/Hijos,Tarifa
Pasajero,,,,
0,29.136361,0.507865,0.379775,34.238473
1,30.271412,0.538117,0.383408,30.174504


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Una posibilidad para agrupar por índice y alguna columna es utilizar `pd.Grouper()`. Por ejemplo: `df.groupby([pd.Grouper(level=0), "columna"])`. Por otra parte, `df.groupby(dropna=False)` crea grupos con la etiqueta `NaN`.

</div>

---

## <font color="#5E5E5E">2. Estructura de la agrupación </font>

### <font color="#5E5E5E">   `DataFrameGroupBy`</font>

El método `DataFrame.groupby()` devuelve un objeto de tipo `DataFrameGroupBy` que permite operar con grupos. ***Una manera de entenderlo*** es como "*una estructura en la que se organiza el `DataFrame` original en varios `DataFrame`, uno por grupo, y que implementa métodos que se aplican internamente sobre cada `DataFrame`* [**(+ inf)**](https://pandas.pydata.org/pandas-docs/stable/reference/groupby.html#computations-descriptive-stats)". 

In [11]:
df_grouped = df_titanic.groupby('Clase')
type(df_grouped)

pandas.api.typing.DataFrameGroupBy

Mediante `DataFrameGroupBy.get_group(group)` es posible *extraer* el `DataFrame` correspondiente a los elementos de cada grupo de la estructura `DataFrameGroupBy`.

In [12]:
df_grouped.get_group('first').head()

,Sobrevive,Clase,Nombre,Sexo,Edad,Esposa/Hermanos,Padres/Hijos,Ticket,Tarifa,Número de Cabina,Puerto
Pasajero,,,,,,,,,,,
2,Yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
4,Yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
7,No,first,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
12,Yes,first,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
24,Yes,first,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,35.5000,A6,S


También es posible iterar sobre la estructura `DataFrameGroupBy` y obtener el `DataFrame` correspondiente a cada grupo.

In [ ]:
for group, df_group in df_grouped:
    print(group) 
    display(df_group.head(3)) # Muestra la cabecera del DataFrame
    #break;                    # Para mostrar solo el primero

Los métodos de `DataFrameGroupBy` "*se aplican*" sobre cada `DataFrame`. 

In [13]:
df_titanic.mean(numeric_only=True)
#df_titanic.groupby('Clase').mean(numeric_only=True)

Edad               29.699118
Esposa/Hermanos     0.523008
Padres/Hijos        0.381594
Tarifa             32.204208
dtype: float64

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
**Importante.** Esta manera de describir la estructura `DataFrameGroupBy` es **poco rigurosa**. Sin embargo, *también es muy útil*. Por otra parte, **no se suele iterar** sobre los  `DataFrameGroupBy`, ni acceder a los `DataFrame` por separado. Pero el ejemplo visto es muy útil para entender la estructura. 
</div>

### <font color="#00586D">`SeriesGroupBy` </font>

El acceso a una columna en un objeto `DataFrameGroupBy` produce un objeto denominado `SeriesGroupBy`,
 en el que los datos están también agrupados con el mismo criterio que en el `DataFrameGroupBy`.

In [14]:
df_titanic.groupby('Clase')['Tarifa']

In [15]:
for grupo, serie in df_titanic.groupby('Clase')['Tarifa']:
    print(grupo)
    display(serie.head(3))
    break;                    # Para mostrar solo la primera

first


Pasajero
2    71.2833
4    53.1000
7    51.8625
Name: Tarifa, dtype: float64

De igual manera, `SeriesGroupBy` implementa métodos que "*actúan sobre la `Series` correspondiente a cada grupo*". El resultado suele ser un objeto de tipo `Series` indexado por grupo.

In [16]:
# df_titanic['Tarifa'].mean()
df_titanic.groupby('Clase')['Tarifa'].mean()

Clase
first     84.154687
second    20.662183
third     13.675550
Name: Tarifa, dtype: float64

Por defecto, el resultado de `DataFrame.groupby()` ordena el resultado por grupos, pero puede reordenarse por valores.

In [17]:
df_titanic.groupby('Clase')['Tarifa'].mean().sort_values(ascending=False)

Clase
first     84.154687
second    20.662183
third     13.675550
Name: Tarifa, dtype: float64

---

## <font color="#5E5E5E">3. Agrupación por categorias creadas</font>


Anteriormente se ha visto que la notación `DataFrame.groupby(columna)` es equivalente a `DataFrame.groupby(DataFrame[columna])`. Esta notación abre la puerta, y ayuda a entender, una funcionalidad útil: agrupar los datos por 
`Series` que no son columnas del `DataFrame` original. 

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i> El único requisito es que el índice de la nueva `Series` debe coincidir con el índice del `DataFrame` original.
</div>

In [18]:
df_titanic.groupby(df_titanic['Clase']).mean(numeric_only=True)

,Edad,Esposa/Hermanos,Padres/Hijos,Tarifa
Clase,,,,
first,38.233441,0.416667,0.356481,84.154687
second,29.877630,0.402174,0.380435,20.662183
third,25.140620,0.615071,0.393075,13.675550


Una de las funcionalidades más comunes a este respecto es la agrupación por `Series` creadas *a partir datos numericos*. Para ello, es necesario crear una función que permita transformar la columna no categórica en categórica. Por ejemplo, la siguiente función, `is_expensive` devuelve `True` cuando el valor de un elemento es mayor que 20. Al aplicarla a la columna `df_titanic['Tarifa']`, se transforma en una columna categórica.

In [19]:
def is_expensive(elemento):
    return elemento>20

is_expensive(df_titanic['Tarifa'])

Pasajero
1      False
2       True
3      False
4       True
5      False
       ...  
887    False
888     True
889     True
890     True
891    False
Name: Tarifa, Length: 891, dtype: bool

Puede utilizarse el resultado de esta operación para hacer una agrupación.

In [20]:
df_titanic.groupby(is_expensive(df_titanic['Tarifa']))['Edad'].mean()  # Muestra la edad media de los pasajeros según el tipo de tarifa

Tarifa
False    28.012583
True     31.740712
Name: Edad, dtype: float64

En general, cuando las funciones son sencillas, se pueden escribir directamente sin perjudicar la legibilidad.

In [22]:
df_titanic.groupby(df_titanic['Tarifa']>20)['Edad'].mean()

Tarifa
False    28.012583
True     31.740712
Name: Edad, dtype: float64

Si la función no se puede vectorizar, es necesario utilizar el método `Series.apply()`.

In [23]:
def disc_edad(elemento):
    if elemento>60:
        return 'Anciano'
    elif elemento<30:
        return 'Joven'
    else:
        return 'Adulto'
    
df_titanic['Edad'].dropna().apply(disc_edad)

Pasajero
1       Joven
2      Adulto
3       Joven
4      Adulto
5      Adulto
        ...  
886    Adulto
887     Joven
888     Joven
890     Joven
891    Adulto
Name: Edad, Length: 714, dtype: str

Es posible agrupar en función de esta nueva categoría

In [24]:
# Tarifa media por cada una de las tres franjas de edad
df_titanic.groupby(df_titanic['Edad'].dropna().apply(disc_edad))['Tarifa'].mean() 

Edad
Adulto     41.058482
Anciano    41.371214
Joven      29.207562
Name: Tarifa, dtype: float64

### <font color="#5E5E5E">  Discretización</font>

En elemplo anterior se ha dividido el rango de valores de la columna `df_titanic['Tarifa']`, y `df_titanic['Edad']`, después, de manera arbitraria. *pandas* proporciona dos funciones para llevar a cabo la discretización.

#### <font color="#5E5E5E"> Igual anchura</font>

Consiste en dividir el rango de valores de una serie en intervalos de igual anchura, es decir, el rango de cada intervalo es similar. La función `pd.cut()` lleva a cabo esta división, y devuelve una nueva `Series` en la que cada valor numérico se sustituye por la etiqueta correspondiente al intervalo que la contiene. Como parámetro toma el número intervalos considerados.

In [25]:
pd.cut(df_titanic['Tarifa'],3) # Discretiza los valores de la columna en 3 intervalos

Pasajero
1      (-0.512, 170.776]
2      (-0.512, 170.776]
3      (-0.512, 170.776]
4      (-0.512, 170.776]
5      (-0.512, 170.776]
             ...        
887    (-0.512, 170.776]
888    (-0.512, 170.776]
889    (-0.512, 170.776]
890    (-0.512, 170.776]
891    (-0.512, 170.776]
Name: Tarifa, Length: 891, dtype: category
Categories (3, interval[float64, right]): [(-0.512, 170.776] < (170.776, 341.553] < (341.553, 512.329]]

Por defecto, la etiqueta asociada a cada intervalo es un *string* que representa el rango de valores que lo caracteriza. El parámetro `labels` permite pasar una lista con los nombres de los intervalos resultantes.

In [26]:
tarifa_disc = pd.cut(df_titanic['Tarifa'],3, labels=['Barata', 'Media', 'Cara'])
tarifa_disc

Pasajero
1      Barata
2      Barata
3      Barata
4      Barata
5      Barata
        ...  
887    Barata
888    Barata
889    Barata
890    Barata
891    Barata
Name: Tarifa, Length: 891, dtype: category
Categories (3, str): ['Barata' < 'Media' < 'Cara']

Es posible agrupar los datos en función del resultado de esta transformación.

In [27]:
df_titanic.groupby(tarifa_disc, observed=True)['Edad'].mean() # Edad media según e tipo de tarifa.

Tarifa
Barata    29.661164
Media     30.333333
Cara      35.333333
Name: Edad, dtype: float64

#### <font color="#5E5E5E"> Igual frecuencia</font>

La discretización en igual anchura presenta un problema. Si se observa el número de valores que caen dentro de cada categoría, se aprecia que el reparto es muy desigual.

In [28]:
tarifa_disc.value_counts()

Tarifa
Barata    871
Media      17
Cara        3
Name: count, dtype: int64

Esto se debe a la existencia de valores extremos, poco representativos, pero que actúan sobre el rango total y sobre los intervalos resultantes. La discretización en igual frecuencia determina el rango de cada intervalo de modo que el número de elementos que caigan en cada uno de ellos sea similar. La función `pd.qcut()` lleva a cabo la discretización en igual frecuencia.

In [29]:
tarifa_disc = pd.qcut(df_titanic['Tarifa'],3)
tarifa_disc

Pasajero
1      (-0.001, 8.662]
2      (26.0, 512.329]
3      (-0.001, 8.662]
4      (26.0, 512.329]
5      (-0.001, 8.662]
            ...       
887      (8.662, 26.0]
888    (26.0, 512.329]
889      (8.662, 26.0]
890    (26.0, 512.329]
891    (-0.001, 8.662]
Name: Tarifa, Length: 891, dtype: category
Categories (3, interval[float64, right]): [(-0.001, 8.662] < (8.662, 26.0] < (26.0, 512.329]]

Puede apreciarse ahora que los rangos correspondientes a cada intervalo son desiguales, pero que el número de elementos que cae dentro de cada uno es similar.

In [30]:
tarifa_disc.value_counts()

Tarifa
(-0.001, 8.662]    308
(26.0, 512.329]    295
(8.662, 26.0]      288
Name: count, dtype: int64

`pd.qcut()` también admite cuantiles para establecer los intervalos.

In [31]:
tarifa_disc = pd.qcut(df_titanic['Tarifa'], q=[0, 0.25, 0.75, 1])
tarifa_disc.value_counts()

Tarifa
(7.91, 31.0]       446
(-0.001, 7.91]     223
(31.0, 512.329]    222
Name: count, dtype: int64

Esta discretización puede utilizarse también para agrupar los datos.

In [32]:
tarifa_disc = pd.qcut(df_titanic['Tarifa'],3, labels=['Barata', 'Media', 'Cara'])
df_titanic.groupby(tarifa_disc, observed=True)['Edad'].mean()

Tarifa
Barata    28.304381
Media     27.997967
Cara      32.456395
Name: Edad, dtype: float64

---

## <font color="#5E5E5E"><i class="fa fa-plus-circle" aria-hidden="true" style="color:#5E5E5E"></i> Eficiencia</font>

El uso del método `DataFrame.groupby()` permite hacer la agrupación de manera muy eficiente. Para ilustrarlo, se calculará el valor medio (`Overall`) de los jugadores de cada club. Con el fin de que se aprecie la diferencia, se utilizarán 2000 registros del conjunto `df_fifa` original.

In [33]:
df_fifa_all = pd.read_csv('./data/fifa19.csv', index_col=0).set_index('Name')
df_fifa_all = df_fifa_all.iloc[:2000]

En el primer caso, se utiliza la selección condicional

In [34]:
%%timeit -n 10
for club in df_fifa_all['Club'].unique():
    avg = df_fifa_all[df_fifa_all['Club']==club]['Overall'].mean()

410 ms ± 3.83 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


La siguiente celda hace la misma operación, pero iterando dobre una estructura `DataFrameGroupBy`.

In [35]:
%%timeit -n 10
for club, df_club in df_fifa_all.groupby('Club'):
    avg = df_club['Overall'].mean()

90.4 ms ± 15.5 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Por último, se lleva a cabo la misma operación, pero con `DataFrame.groupby()`.

In [36]:
%%timeit -n 10
avg = df_fifa_all.groupby('Club')['Overall'].mean()

749 μs ± 148 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>